# TripletNet Evaluation

## Import Libraries

In [1]:
!pip install datasets==3.6.0 --no-warn-conflicts --quiet
!pip install faiss-cpu==1.11.0 --no-warn-conflicts --quiet
!pip install torchmetrics --quiet

import os
import json
from pathlib import Path
import datasets
import faiss
import torchmetrics
import numpy as np
import random
import pandas as pd
from tqdm import tqdm
from google.colab import userdata

import torch
import torchvision
import huggingface_hub
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.5/961.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Configuration

In [2]:
import os
from pathlib import Path
import torch

class Config:
    """
    Configuration class for the Triplet Network training project.
    """

    # ========== Project Name & Paths ==========
    project_name = "logo_recognition_similarity_search_project"
    base_dir = Path("/content/drive/MyDrive") / project_name
    output_dir = base_dir / "output_last"
    cache_dir = base_dir / "cache"

    # ========== Model Settings ==========
    backbone_type = "Resnet50"     # Options: "Resnet50", "Vgg16", "Efficientnet"
    model_id = f'mlproject5606/Logo-Recognition-{backbone_type}-TripletLoss'
    model_output_dir = output_dir / f'{model_id.split("/")[1]}'
    model_evaluation_dir = model_output_dir / "evaluation"


    # ========== Dataset Settings ==========
    dataset_path = f"mlproject5606/{backbone_type}-TripletLoss-Embedding-Dataset"
    dataset_cache_dir = cache_dir / dataset_path.split("/")[-1]

    # ========== Runtime Settings ==========
    device = "cuda" if torch.cuda.is_available() else "cpu"
    seed = 42
    batch_size = 512
    num_workers = 4

    # ========== Evaluation parameters ==========
    top_ks = [1, 5, 10]
    faiss_index_column = "embedding"

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(cache_dir, exist_ok=True)
    os.makedirs(model_evaluation_dir, exist_ok=True)


# Create configuration object
cfg = Config()

## Load Embeddings Dataset

In [3]:
def load_dataset_with_embeddings(dataset_path, cache_dir=None):
    ds = datasets.load_dataset(path = dataset_path,cache_dir=cache_dir)
    ds_train = ds['train'].with_format("numpy")
    ds_test = ds['test'].with_format("numpy")
    return ds_train, ds_test

## Evaluation

In [8]:
import torch
from torchmetrics.retrieval import (
    RetrievalPrecision,
    RetrievalRecall,
    RetrievalMAP,
    RetrievalMRR,
    RetrievalNormalizedDCG,
)

def evaluate_model_with_faiss(
    model_name,
    ds_train,
    ds_test,
    embedding_column="embedding",
    label_column="category",
    top_ks=[1, 5, 10],
    batch_size=100,
    save_csv_path=None
):
    max_k = max(top_ks)
    all_preds = []
    all_targets = []
    all_indexes = []
    ds_train = ds_train.add_faiss_index(column=embedding_column)

    def compute_metrics_batch(batch, indices):
        queries = batch[embedding_column]
        true_labels = batch[label_column]

        scores, retrieved = ds_train.get_nearest_examples_batch(
            embedding_column, queries=queries, k=max_k
        )

        preds, targets, query_ids = [], [], []
        for i, (score_list, retrieved_batch) in enumerate(zip(scores, retrieved)):
            qid = indices[i]
            retrieved_labels = retrieved_batch[label_column]
            for s, lbl in zip(score_list, retrieved_labels):
                preds.append(s)
                targets.append(1 if lbl == true_labels[i] else 0)
                query_ids.append(qid)

        return {
            "preds": preds,
            "targets": targets,
            "indexes": query_ids
        }

    mapped = ds_test.map(
        compute_metrics_batch,
        with_indices=True,
        batched=True,
        batch_size=batch_size,
        num_proc=4,
        desc=f"[{model_name}] Evaluating",
        remove_columns=ds_test.column_names,
    )

    preds_tensor = torch.tensor(mapped["preds"], dtype=torch.float32)
    targets_tensor = torch.tensor(mapped["targets"], dtype=torch.int)
    indexes_tensor = torch.tensor(mapped["indexes"], dtype=torch.long)

    results = []
    for k in top_ks:
        p = RetrievalPrecision(top_k=k)(preds_tensor,
                                        targets_tensor,
                                        indexes_tensor
                                        ).item()

        r = RetrievalRecall(top_k=k)(preds_tensor,
                                    targets_tensor,
                                    indexes_tensor
                                    ).item()

        f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0

        results.append({
            "Model": model_name,
            "Top-K": k,
            "Precision@K": p,
            "Recall@K": r,
            "F1@K": f1,
        })

    df = pd.DataFrame(results)
    if save_csv_path:
        df.to_csv(save_csv_path, index=False)
        mapped.save_to_disk(f"{str(save_csv_path).split('.')[0]}_mapped")

    return df


### Efficientnet

In [ ]:
save_csv_eff_path = cfg.output_dir/"Logo-Recognition-Efficientnet-TripletLoss/evaluation/evaluation_metrics.csv"

In [6]:
ds_train_eff, ds_test_eff = load_dataset_with_embeddings("mlproject5606/Efficientnet-TripletLoss-Embedding-Dataset")

README.md:   0%|          | 0.00/700 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/57 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/57 [00:00<?, ?it/s]

train-00000-of-00057.parquet:   0%|          | 0.00/507M [00:00<?, ?B/s]

train-00001-of-00057.parquet:   0%|          | 0.00/509M [00:00<?, ?B/s]

...

test-00015-of-00016.parquet:   0%|          | 0.00/504M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1279860 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/142207 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/355517 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/57 [00:00<?, ?it/s]

In [9]:
results_eff = evaluate_model_with_faiss(model_name="EfficientNet",
                                        ds_train=ds_train_eff,
                                        ds_test=ds_test_eff,
                                        embedding_column=cfg.faiss_index_column,
                                        label_column="category",
                                        top_ks=cfg.top_ks,
                                        batch_size=cfg.batch_size,
                                        save_csv_path=save_csv_eff_path
                                        )

results_eff

  0%|          | 0/1280 [00:00<?, ?it/s]

,Model,Top-K,Precision@K,Recall@K,F1@K
0,EfficientNet,1,0.180987,0.045288,0.072447
1,EfficientNet,5,0.181097,0.226684,0.201343
2,EfficientNet,10,0.182856,0.462976,0.262167


In [10]:
results_eff = pd.read_csv(save_csv_eff_path)
results_eff

,Model,Top-K,Precision@K,Recall@K,F1@K
0,EfficientNet,1,0.180987,0.045288,0.072447
1,EfficientNet,5,0.181097,0.226684,0.201343
2,EfficientNet,10,0.182856,0.462976,0.262167


## Disconnect

In [ ]:
# Disconnect and delete the current runtime in Google Colab
from google.colab import runtime

runtime.unassign()